# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata (not as a dict—use .name/.description attributes!)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Keywords: {dataset.metadata.keywords}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Here, we'll enumerate all the record sets defined in the dataset, along with their fields and columns. All entities are referenced by their `@id`.

In [ ]:
# Retrieve record sets and their fields. All by @id.
record_sets = dataset.record_sets()
print("Record Sets found in the dataset:")
for rs in record_sets:
    print(f"Record Set: {rs['@id']} | Name: {rs.get('name', 'N/A')}")
    fields = rs.get('field', [])
    # Ensure fields is always a list
    if not isinstance(fields, list):
        fields = [fields]
    print("  Fields:")
    for f in fields:
        if isinstance(f, dict):
            print(f"    Field @id: {f.get('@id', '')} | Name: {f.get('name', '')}")
        else:
            print(f"    Field @id: {f}")
    columns = rs.get('column', [])
    if not isinstance(columns, list):
        columns = [columns]
    print("  Columns:")
    for col in columns:
        if isinstance(col, dict):
            print(f"    Column @id: {col.get('@id', '')} | Name: {col.get('name', '')}")
        else:
            print(f"    Column @id: {col}")
    print("-")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set by @id
record_sets_ids = [rs['@id'] for rs in dataset.record_sets()]
dataframes = {}
for record_set_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for RecordSet {record_set_id}, columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# For demonstration, pick the first record set if available
if record_sets_ids:
    first_record_set_id = record_sets_ids[0]
    print(f"First few rows for record set {first_record_set_id}:")
    display(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

All operations reference fields by their `@id`.

In [ ]:
# For demonstration, let's select a numeric field (@id) and group field (@id) from the first record set
first_rs_id = record_sets_ids[0] if record_sets_ids else None
df = dataframes[first_rs_id] if first_rs_id in dataframes else None

# Inspect columns to choose appropriate fields
if df is not None:
    print("Available columns (fields @id):", df.columns.tolist())
    # Try to find a likely numeric field: 'phq9_score', 'gad7_score', or any that match
    numeric_field_candidates = [col for col in df.columns if 'score' in col or 'age' in col]
    numeric_field_id = numeric_field_candidates[0] if numeric_field_candidates else df.columns[0]
    # Try to find a group field: e.g. 'village', 'gender', or any with categorical values
    group_field_candidates = [col for col in df.columns if 'village' in col or 'gender' in col or 'education' in col]
    group_field = group_field_candidates[0] if group_field_candidates else None

    # Apply filtering (e.g., scores greater than threshold)
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a field if available
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field} (average {numeric_field_id}):")
        display(grouped_df.head())
else:
    print("No DataFrame loaded or no columns available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of a numeric field and compare group averages, referencing columns by their `@id`.

In [ ]:
# Visualization:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20)
    plt.title(f"Distribution of {numeric_field_id} (field @id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"Average {numeric_field_id} by {group_field} (fields @id)")
        plt.ylabel(f"{numeric_field_id} (mean)")
        plt.xlabel(group_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset from Kilifi County, Kenya provides structured, AI-ready survey data with demographic and psychometric indicators.
- Using `mlcroissant`, we loaded metadata and reviewed record sets, referencing all entities by their `@id`.
- Numeric and categorical field analyses reveal data distributions and notable differences between demographic subgroups.
- The dataset demonstrates clear FAIR and Croissant standards, making it ready for further analysis or ML applications.